In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env file

True

# Loading libraries

In [3]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
import feedparser
from urllib.parse import quote

# Defining tools

In [48]:
ARXIV_API = "http://export.arxiv.org/api/query"

In [49]:
@tool
def search_arxiv(
    query,
    max_results=5,
    start=0
):
    """Search for research papers on arXiv based on a query."""
    
    query = quote(query)
    url = (
        f"{ARXIV_API}?"
        f"search_query=all:{query}"
        f"&start={start}"
        f"&max_results={max_results}"
        f"&sortBy=submittedDate"
        f"&sortOrder=descending"
    )
    # print(url)
    feed = feedparser.parse(url)

    results = []
    for entry in feed.entries:
        results.append({
            "id": entry.id,
            "title": entry.title.strip().replace("\n", " "),
            "published": entry.published
        })
    return results

In [44]:
@tool
def fetch_abstract(arxiv_id):
    """
    Fetch the abstract of a research paper from arXiv given its ID.
    """
    url = f"{ARXIV_API}?id_list={arxiv_id.split('/')[-1]}"
    feed = feedparser.parse(url)

    if not feed.entries:
        return None

    entry = feed.entries[0]
    return entry.summary.strip().replace("\n", " ")

# LLM and system prompts

In [45]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [80]:
PAPER_CRAWLER_SYSTEM_PROMPT = """
You are an AI assistant that helps users find research papers from arXiv.
Use the `search_arxiv` tool to search for papers.

IMPORTANT: Always format your response as a Python dictionary with arxiv_id as keys and paper titles as values.
Example output:
{
    "http://arxiv.org/abs/2301.00001": "Title of First Paper",
    "http://arxiv.org/abs/2301.00002": "Title of Second Paper"
}

Return ONLY the dictionary, no other text.
"""

In [47]:
PAPER_SUMMARIZER_SYSTEM_PROMPT = """
You are an AI assistant that summarizes research papers from arXiv.
Use the `fetch_abstract` tool to fetch the abstract of a paper.
"""

In [52]:
PAPER_CLAIM_EXTRACTOR_SYSTEM_PROMPT = """
You are an AI assistant that extracts claims from a paper summary.
Identify and list the key claims made in the summary.
"""

In [53]:
PAPER_CLAIM_VERIFIER_SYSTEM_PROMPT = """
You are an AI assistant that verifies claims from a paper summary.
Identfy whether each claim is supported, refuted, or not enough information is available in the summary.
"""

In [54]:
FINDING_SYNTHESIZER_SYSTEM_PROMPT = """
You are an AI assistant that synthesizes findings from multiple research paper summaries.
"""

# Define the Agents

In [81]:
from pydantic import BaseModel
from typing import Dict

class PaperResult(BaseModel):
    """Key-value pair format for paper results"""
    papers: Dict[str, str]  # key: arxiv_id, value: paper_title
    total_count: int

class CrawlerOutput(BaseModel):
    query: str
    results: PaperResult

In [78]:
def make_paper_crawler_agent():
    return create_agent(
        model=llm,
        tools=[search_arxiv],
        system_prompt=PAPER_CRAWLER_SYSTEM_PROMPT,
    )

In [51]:
def make_summarizer_agent():
    return create_agent(
        model=llm,
        tools=[fetch_abstract],
        system_prompt=PAPER_SUMMARIZER_SYSTEM_PROMPT 
    )

In [55]:
def make_claim_extractor_agent():
    return create_agent(
        model=llm,
        system_prompt=PAPER_CLAIM_EXTRACTOR_SYSTEM_PROMPT 
    )

In [56]:
def make_claim_verifier_agent():
    return create_agent(
        model=llm,
        system_prompt=PAPER_CLAIM_VERIFIER_SYSTEM_PROMPT 
    )

In [57]:
def make_result_synthesizer_agent():
    return create_agent(
        model=llm,
        system_prompt=FINDING_SYNTHESIZER_SYSTEM_PROMPT 
    )

# Creating Agents

In [82]:
search_agent = make_paper_crawler_agent()
summarizer_agent = make_summarizer_agent()
claim_extractor_agent = make_claim_extractor_agent()
claim_verifier_agent = make_claim_verifier_agent()
result_synthesizer_agent = make_result_synthesizer_agent()

In [86]:
import ast

In [129]:
def get_relevant_papers(query):
    response = search_agent.invoke({"messages":query, "role":"user"})
    result_text = response["messages"][-1].content
    try:
        # Parse the dictionary response
        import ast
        papers_dict = ast.literal_eval(result_text.strip())
        
        return {
            "query": query,
            "results": {
                "papers": papers_dict,
                "total_count": len(papers_dict)
            }
        }
    except (ValueError, SyntaxError):
        return {
            "query": query,
            "results": {
                "papers": {},
                "total_count": 0,
                "error": "Failed to parse results"
            }
        }

In [130]:
search_results = get_relevant_papers("HCI")

In [122]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio

async def process_papers_parallel(papers, agent, max_workers=5):
    """Process multiple papers in parallel using aysncio"""
    tasks = [agent.ainvoke({"input": paper}) for paper in papers]
    results = await asyncio.gather(*tasks)
    return results

In [131]:
paper_entries = search_results["results"]["papers"]

In [132]:
paper_entries.keys()

dict_keys(['http://arxiv.org/abs/2601.10025v1', 'http://arxiv.org/abs/2601.08640v1', 'http://arxiv.org/abs/2601.06935v1', 'http://arxiv.org/abs/2601.06402v1', 'http://arxiv.org/abs/2601.05871v2'])

In [125]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
results = asyncio.run(process_papers_parallel(paper_entries.keys(), summarizer_agent, max_workers=5))

In [134]:
import json

In [136]:
paper_entries.keys()

dict_keys(['http://arxiv.org/abs/2601.10025v1', 'http://arxiv.org/abs/2601.08640v1', 'http://arxiv.org/abs/2601.06935v1', 'http://arxiv.org/abs/2601.06402v1', 'http://arxiv.org/abs/2601.05871v2'])

In [137]:
sample_key="http://arxiv.org/abs/2601.10025v1"

In [138]:
summary_result = summarizer_agent.invoke({"messages":sample_key,"role":"user"})

In [141]:
summary = summary_result["messages"][-1].content

In [144]:
claim_result = claim_extractor_agent.invoke({"messages": summary, "role":"user"})

In [149]:
sample_claim = claim_result["messages"][-1].content

# Capture Paper claims

In [146]:
def collect_paper_claims(paper_entries, summarizer_agent, claim_extractor_agent):
    claims_dict = {}
    summary_dict = {}
    for paper in paper_entries.keys():
        summary_result = summarizer_agent.invoke({"messages":paper,"role":"user"})
        summary = summary_result["messages"][-1].content
        summary_dict[paper] = summary
        claim_result = claim_extractor_agent.invoke({"messages": summary, "role":"user"})
        claims = claim_result["messages"][-1].content
        claims_dict[paper] = claims
    return summary_dict, claims_dict

In [147]:
summary_dict, claims_dict = collect_paper_claims(paper_entries, summarizer_agent, claim_extractor_agent)

# Verify Paper Claims

In [151]:
verify_result = claim_verifier_agent.invoke({"messages": sample_claim +" "+summary, "role":"user"})

In [152]:
verify_result["messages"][-1].content

'1. Supported\n2. Supported\n3. Supported\n4. Supported\n5. Supported\n6. Supported\n7. Supported'

In [153]:
def verify_paper_claims(paper_entries, summary_dict, claims_dict, claim_verifier_agent):
    verified_claims_dict = {}
    for paper in paper_entries.keys():
        summary = summary_dict[paper]
        claims = claims_dict[paper]
        claim_result = claim_verifier_agent.invoke({"messages": claims +" "+summary, "role":"user"})
        verified_claims = claim_result["messages"][-1].content
        verified_claims_dict[paper] = verified_claims
    return verified_claims_dict

In [154]:
verify_dict = verify_paper_claims(paper_entries, summary_dict, claims_dict, claim_verifier_agent)

In [157]:
for paper in paper_entries:
    title = paper_entries[paper]
    summary = summary_dict[paper]
    claims = claims_dict[paper]
    verification = verify_dict[paper]
    line = f"Title: {title} \n Summary: {summary} \n Claim: {claims} \n Verification: {verification}"
    print(line)
    print("==================")
    

Title: Structured Personality Control and Adaptation for LLM Agents 
 Summary: The paper discusses the role of Large Language Models (LLMs) in human-computer interaction (HCI), focusing on their potential to exhibit human-like characteristics, particularly personality. It introduces a framework for modeling LLM personality based on Jungian psychological types, which includes three mechanisms: a dominant-auxiliary coordination for coherent expression, a reinforcement-compensation mechanism for context adaptation, and a reflection mechanism for long-term personality evolution. This design enables LLMs to maintain nuanced traits while adapting to interaction demands. The effectiveness of personality alignment is assessed using Myers-Briggs Type Indicator questionnaires and various challenge scenarios. The findings indicate that personality-aware LLMs can facilitate coherent and context-sensitive interactions, enhancing naturalistic agent design in HCI. 
 Claim: 1. Large Language Models (L

# Markdown 2 PDF

In [9]:
import markdown2
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch

In [10]:
def markdown_to_pdf(md_file, pdf_file):
    """Convert markdown file to PDF using markdown2 and reportlab"""
    # Read markdown file
    with open(md_file, 'r', encoding='utf-8') as f:
        md_content = f.read()
    
    # Convert markdown to HTML
    html_content = markdown2.markdown(md_content)
    
    # Create PDF using reportlab
    doc = SimpleDocTemplate(pdf_file, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []
    
    # Parse HTML and add to story (simplified approach)
    # Remove HTML tags for reportlab compatibility
    import re
    text = re.sub('<[^<]+?>', '', html_content)
    
    for line in text.split('\n'):
        if line.strip():
            story.append(Paragraph(line, styles['Normal']))
            story.append(Spacer(1, 0.2*inch))
    
    doc.build(story)
    print(f"PDF created: {pdf_file}")

# Convert the markdown file
markdown_to_pdf('SLROutput/AgenticAIOrchestration.md', 'SLROutput/AgenticAIOrchestration.pdf')

PDF created: SLROutput/AgenticAIOrchestration.pdf
